# Built-in Algorithms

## Overview of Built-in Algorithms

SageMaker built-in algorithms are pre-optimized for performance and scalability. They handle distributed training automatically, support spot instances for cost savings, and integrate seamlessly with SageMaker's training infrastructure.

## XGBoost for Regression and Classification

In [ ]:
from sagemaker.estimator import Estimator
import sagemaker

session = sagemaker.Session()
role = 'arn:aws:iam::123456789012:role/SageMakerRole'
bucket = session.default_bucket()

# XGBoost container URI
xgboost_container = '246618743249.dkr.ecr.us-east-1.amazonaws.com/sagemaker-xgboost:1.5-1'

# Create XGBoost estimator
xgb_estimator = Estimator(
    image_uri=xgboost_container,
    role=role,
    instance_count=1,
    instance_type='ml.m5.xlarge',
    output_path=f's3://{bucket}/xgboost-output',
    sagemaker_session=session
)

# Set hyperparameters
xgb_estimator.set_hyperparameters(
    objective='binary:logistic',
    num_round=100,
    max_depth=5,
    eta=0.2,
    gamma=4,
    min_child_weight=6,
    subsample=0.8
)

# Train the model
xgb_estimator.fit(
    {'training': f's3://{bucket}/train-data.csv'},
    job_name='xgboost-training-job'
)

## Linear Learner for Large-Scale Problems

In [ ]:
from sagemaker.linear_learner import LinearLearner

linear_learner = LinearLearner(
    role=role,
    instance_count=1,
    instance_type='ml.m5.xlarge',
    output_path=f's3://{bucket}/linear-output',
    sagemaker_session=session
)

# Set hyperparameters
linear_learner.set_hyperparameters(
    feature_dim=100,
    mini_batch_size=32,
    predictor_type='binary_classifier',
    loss='logistic',
    optimizer='adam',
    learning_rate=0.01,
    epochs=10
)

# Train
linear_learner.fit(
    {'training': f's3://{bucket}/train-data.recordio'},
    job_name='linear-learner-job'
)

## Image Classification with ResNet

In [ ]:
from sagemaker.image_uris import retrieve

# Get Image Classification container
image_uri = retrieve(
    framework='image-classification',
    region='us-east-1',
    version='latest'
)

image_classifier = Estimator(
    image_uri=image_uri,
    role=role,
    instance_count=1,
    instance_type='ml.p3.2xlarge',
    output_path=f's3://{bucket}/image-output',
    sagemaker_session=session
)

# Set hyperparameters
image_classifier.set_hyperparameters(
    num_classes=10,
    num_layers=50,
    image_shape='3,224,224',
    epochs=30,
    learning_rate=0.01,
    batch_size=32,
    optimizer='sgd'
)

# Train
image_classifier.fit(
    {'training': f's3://{bucket}/image-train/'},
    job_name='image-classification-job'
)

## BlazingText for NLP Tasks

In [ ]:
from sagemaker.blazingtext import BlazingText

blazingtext = BlazingText(
    role=role,
    instance_count=1,
    instance_type='ml.p3.2xlarge',
    output_path=f's3://{bucket}/blazingtext-output',
    sagemaker_session=session
)

# Set hyperparameters for text classification
blazingtext.set_hyperparameters(
    mode='supervised',
    epochs=5,
    learning_rate=0.05,
    word_ngrams=2,
    vector_dim=100,
    batch_size=32
)

# Train
blazingtext.fit(
    {'training': f's3://{bucket}/text-train.txt'},
    job_name='blazingtext-job'
)

## Algorithm Selection Guide

```json
{
  "algorithm_selection": {
    "xgboost": {
      "use_cases": ["Tabular data", "Classification", "Regression"],
      "strengths": ["Fast", "Handles missing values", "Feature importance"],
      "input_format": "CSV or LibSVM"
    },
    "linear_learner": {
      "use_cases": ["Large datasets", "Linear relationships"],
      "strengths": ["Scalable", "Fast inference"],
      "input_format": "RecordIO or CSV"
    },
    "image_classification": {
      "use_cases": ["Image recognition", "Object detection"],
      "strengths": ["Pre-trained models", "Transfer learning"],
      "input_format": "RecordIO or image files"
    },
    "blazingtext": {
      "use_cases": ["Text classification", "Word embeddings"],
      "strengths": ["Fast training", "Supports multiple languages"],
      "input_format": "Text files"
    }
  }
}
```

## Deploying Built-in Algorithm Models

In [ ]:
# Deploy XGBoost model
predictor = xgb_estimator.deploy(
    initial_instance_count=1,
    instance_type='ml.m5.large',
    endpoint_name='xgboost-endpoint'
)

# Make predictions
import csv
import io

# Prepare test data
test_data = '5.1,3.5,1.4,0.2'

# Invoke endpoint
response = predictor.predict(test_data)
print(f"Prediction: {response}")

## Quiz 1

<div class="quiz" data-correct="0">
  <p class="font-semibold mb-3">❓ Which algorithm is best for tabular data classification?</p>
  <div class="space-y-2">
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q8374920" value="0">
      <span>XGBoost</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q8374920" value="1">
      <span>Image Classification</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q8374920" value="2">
      <span>BlazingText</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q8374920" value="3">
      <span>Linear Learner only</span>
    </label>
  </div>
  <button class="quiz-btn mt-3 px-4 py-2 bg-blue-600 text-white rounded text-sm font-medium hover:bg-blue-700">Check Answer</button>
  <p class="quiz-result text-sm mt-2 hidden"></p>
</div>

## Quiz 2

<div class="quiz" data-correct="2">
  <p class="font-semibold mb-3">❓ What is BlazingText primarily used for?</p>
  <div class="space-y-2">
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q5729384" value="0">
      <span>Image classification</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q5729384" value="1">
      <span>Time series forecasting</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q5729384" value="2">
      <span>Text classification and embeddings</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q5729384" value="3">
      <span>Anomaly detection</span>
    </label>
  </div>
  <button class="quiz-btn mt-3 px-4 py-2 bg-blue-600 text-white rounded text-sm font-medium hover:bg-blue-700">Check Answer</button>
  <p class="quiz-result text-sm mt-2 hidden"></p>
</div>

## Quiz 3

<div class="quiz" data-correct="1">
  <p class="font-semibold mb-3">❓ Which algorithm is best for large-scale linear problems?</p>
  <div class="space-y-2">
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q3847291" value="0">
      <span>XGBoost</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q3847291" value="1">
      <span>Linear Learner</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q3847291" value="2">
      <span>Image Classification</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q3847291" value="3">
      <span>BlazingText</span>
    </label>
  </div>
  <button class="quiz-btn mt-3 px-4 py-2 bg-blue-600 text-white rounded text-sm font-medium hover:bg-blue-700">Check Answer</button>
  <p class="quiz-result text-sm mt-2 hidden"></p>
</div>

## Quiz 4

<div class="quiz" data-correct="0">
  <p class="font-semibold mb-3">❓ What input format does XGBoost accept?</p>
  <div class="space-y-2">
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q7291847" value="0">
      <span>CSV or LibSVM</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q7291847" value="1">
      <span>Only RecordIO</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q7291847" value="2">
      <span>Only JSON</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q7291847" value="3">
      <span>Only Parquet</span>
    </label>
  </div>
  <button class="quiz-btn mt-3 px-4 py-2 bg-blue-600 text-white rounded text-sm font-medium hover:bg-blue-700">Check Answer</button>
  <p class="quiz-result text-sm mt-2 hidden"></p>
</div>

## Quiz 5

<div class="quiz" data-correct="2">
  <p class="font-semibold mb-3">❓ Which instance type is recommended for Image Classification training?</p>
  <div class="space-y-2">
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q9284756" value="0">
      <span>ml.t3.medium</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q9284756" value="1">
      <span>ml.m5.large</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q9284756" value="2">
      <span>ml.p3.2xlarge (GPU)</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q9284756" value="3">
      <span>ml.c5.xlarge</span>
    </label>
  </div>
  <button class="quiz-btn mt-3 px-4 py-2 bg-blue-600 text-white rounded text-sm font-medium hover:bg-blue-700">Check Answer</button>
  <p class="quiz-result text-sm mt-2 hidden"></p>
</div>